In [3]:
#import + session DB + subgraph storage obj

import pandas as pd

from HOGDB.graph.subgraph import Subgraph          # constructeur Subgraph (API HO-GDB)
from HOGDB.graph.graph_with_subgraph_storage import GraphwithSubgraphStorage
from HOGDB.graph.graph_with_tuple_storage import *
from HOGDB.graph.edge import Edge
from HOGDB.db.neo4j import Neo4jDatabase

# INIT DB

db = Neo4jDatabase()
gs = GraphwithSubgraphStorage(db)

In [ ]:
#HOGDB do not add indexes automatically, if you want to add indexes, please execute the below queries : 
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Person) REQUIRE n.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Post) REQUIRE n.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Tag) REQUIRE n.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Place) REQUIRE n.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Forum) REQUIRE n.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Comment) REQUIRE n.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Organisation) REQUIRE n.id IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:TagClass) REQUIRE n.id IS UNIQUE;



In [ ]:
#funct to avoid the  problème  of  ' in the text values of properties

def escape_cypher_string(value):
    if isinstance(value, str):
        return value.replace("'", "\\'")
    return value


In [ ]:
#just to retreive the cache Person nodes if you restarted the kernel, you will find this cell before each type node add

PERSON_CSV = "data/person_0_0.csv"

print("== Insert Person nodes ==")

person_cache = {}
person_buffer = []

person_df = pd.read_csv(
    PERSON_CSV,
    sep="|",
    header=0,
    engine="python",
    quotechar='"',
    escapechar='\\',
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in person_df.iterrows():
    node = Node(
        labels=[Label("Person")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("firstName", str, escape_cypher_string(row["firstName"])),
            Property("lastName", str, escape_cypher_string(row["lastName"])),
            Property("gender", str, row["gender"]),
            Property("birthday", int, int(row["birthday"])),
            Property("creationDate", int, int(row["creationDate"])),
            Property("locationIP", str, row["locationIP"]),
            Property("browserUsed", str, row["browserUsed"])
        ]
    )

    person_cache[int(row["id"])] = node
    

print(f"== Total Person nodes inserted: {count} ==")


In [ ]:
#add person nodes 

import pandas as pd
csv_path = "data/person_0_0.csv"

def insert_person_nodes(csv_path, gs, batch_size=1000):
    print("== Insert Person nodes ==")

    person_cache = {}
    person_buffer = []

    person_df = pd.read_csv(
        csv_path,
        sep="|",
        header=0,
        engine="python",
        quotechar='"',
        escapechar='\\',
        encoding="utf-8"
    )

    count = 0

    for _, row in person_df.iterrows():
        node = Node(
            labels=[Label("Person")],
            properties=[
                Property("id", int, int(row["id"])),
                Property("firstName", str, escape_cypher_string(row["firstName"])),
                Property("lastName", str, escape_cypher_string(row["lastName"])),
                Property("gender", str, row["gender"]),
                Property("birthday", int, int(row["birthday"])),
                Property("creationDate", int, int(row["creationDate"])),
                Property("locationIP", str, row["locationIP"]),
                Property("browserUsed", str, row["browserUsed"])
            ]
        )

        person_cache[int(row["id"])] = node
        person_buffer.append(node)

        if len(person_buffer) == batch_size:
            for n in person_buffer:
                gs.add_node(n)

            count += batch_size
            print(f"  committed {count} Person nodes")
            person_buffer.clear()

    # last batch
    if person_buffer:
        for n in person_buffer:
            gs.add_node(n)

        count += len(person_buffer)

    print(f"== Total Person nodes inserted: {count} ==")

    return person_cache
person_cache = insert_person_nodes(csv_path, gs, batch_size=1000)

In [10]:
# cache Post nodes via HOGDB

POST_CSV = "data/post_0_0.csv"

print("== Insert Post nodes ==")

post_cache = {}
post_buffer = []

post_df = pd.read_csv(
    POST_CSV,
    sep="|",
    header=0,
    engine="python",
    quotechar='"',
    escapechar='\\',
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in post_df.iterrows():
    node = Node(
        labels=[Label("Post")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("imageFile", str, row["imageFile"]),
            Property("content", str, escape_cypher_string(row["content"])),
            Property("creationDate", int, int(row["creationDate"])),
            Property("locationIP", str, row["locationIP"]),
            Property("browserUsed", str, row["browserUsed"]),
            Property("language", str, row["language"]),
            Property("length", int, int(row["length"]))
        ]
    )

    post_cache[int(row["id"])] = node


print(f"== Total Post nodes inserted: {count} ==")


== Insert Post nodes ==
== Total Post nodes inserted: 0 ==


In [9]:
# add Post nodes via HOGDB

POST_CSV = "data/post_0_0.csv"

print("== Insert Post nodes ==")

post_cache = {}
post_buffer = []

post_df = pd.read_csv(
    POST_CSV,
    sep="|",
    header=0,
    engine="python",
    quotechar='"',
    escapechar='\\',
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in post_df.iterrows():
    node = Node(
        labels=[Label("Post")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("imageFile", str, row["imageFile"]),
            Property("creationDate", int, int(row["creationDate"])),
            Property("content", str, escape_cypher_string(row["content"])),
            Property("locationIP", str, row["locationIP"]),
            Property("browserUsed", str, row["browserUsed"]),
            Property("language", str, row["language"]),
            Property("length", int, int(row["length"]))
        ]
    )

    post_cache[int(row["id"])] = node
    post_buffer.append(node)

    if len(post_buffer) == NODE_BATCH:
        for n in post_buffer:
            gs.add_node(n)

        count += NODE_BATCH
        print(f"  committed {count} Post nodes")
        post_buffer.clear()

# last batch
if post_buffer:
    for n in post_buffer:
        gs.add_node(n)

    count += len(post_buffer)

print(f"== Total Post nodes inserted: {count} ==")


== Insert Post nodes ==
  committed 1000 Post nodes
  committed 2000 Post nodes
  committed 3000 Post nodes
  committed 4000 Post nodes
  committed 5000 Post nodes
  committed 6000 Post nodes
  committed 7000 Post nodes
  committed 8000 Post nodes
  committed 9000 Post nodes
  committed 10000 Post nodes
  committed 11000 Post nodes
  committed 12000 Post nodes
  committed 13000 Post nodes
  committed 14000 Post nodes
  committed 15000 Post nodes
  committed 16000 Post nodes
  committed 17000 Post nodes
  committed 18000 Post nodes
  committed 19000 Post nodes
  committed 20000 Post nodes
  committed 21000 Post nodes
  committed 22000 Post nodes
  committed 23000 Post nodes
  committed 24000 Post nodes
  committed 25000 Post nodes
  committed 26000 Post nodes
  committed 27000 Post nodes
  committed 28000 Post nodes
  committed 29000 Post nodes
  committed 30000 Post nodes
  committed 31000 Post nodes
  committed 32000 Post nodes
  committed 33000 Post nodes
  committed 34000 Post nodes

In [10]:
type(post_cache)

dict

In [11]:
#get comment_cache

# my paths
COMMENT_CSV = "data/comment_0_0.csv"

NODE_BATCH = 1000






# here  A CACHE

comment_cache = {}   # dict key : id , val : Node



# here  buffers

# here  buffers
comment_buffer = []
tag_buffer = []      
edge_buffer = []   
print("== Insert Comment nodes ==")

comment_df = pd.read_csv(
    COMMENT_CSV,
    sep="|",
    header=0,
    engine="python",
    quotechar='"',
    escapechar='\\',
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0
comment_buffer = []

for _, row in comment_df.iterrows():
    node = Node(
        labels=[Label("Comment")],   # _node sera ajouté par HO-GDB
        properties=[
            Property("id", int, int(row["id"])),
            Property("creationDate", int, int(row["creationDate"])),
            Property("content", str, escape_cypher_string(row["content"])),
            Property("locationIP", str, row["locationIP"]),
            Property("browserUsed", str, row["browserUsed"]),
            Property("length", int, int(row["length"]))
        ]
    )

    comment_cache[int(row["id"])] = node
    



print(f"== Total Comment nodes inserted: {count} ==")


== Insert Comment nodes ==
== Total Comment nodes inserted: 0 ==


In [14]:
#add comment nodes via HOGDB

# my paths
COMMENT_CSV = "data/comment_0_0.csv"

NODE_BATCH = 1000






# here  A CACHE

comment_cache = {}   # dict key : id , val : Node



# here  buffers

# here  buffers
comment_buffer = []
tag_buffer = []      
edge_buffer = []   
print("== Insert Comment nodes ==")

comment_df = pd.read_csv(
    COMMENT_CSV,
    sep="|",
    header=0,
    engine="python",
    quotechar='"',
    escapechar='\\',
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0
comment_buffer = []

for _, row in comment_df.iterrows():
    node = Node(
        labels=[Label("Comment")],   # _node sera ajouté par HO-GDB
        properties=[
            Property("id", int, int(row["id"])),
            Property("creationDate", int, int(row["creationDate"])),
            Property("content", str, escape_cypher_string(row["content"])),
            Property("locationIP", str, row["locationIP"]),
            Property("browserUsed", str, row["browserUsed"]),
            Property("length", int, int(row["length"]))
        ]
    )

    comment_cache[int(row["id"])] = node
    comment_buffer.append(node)

    # batch
    if len(comment_buffer) == NODE_BATCH:
        for n in comment_buffer:
            gs.add_node(n)

                     
        count += NODE_BATCH
        print(f"  committed {count} Comment nodes")
        comment_buffer.clear()

# dernier batch
if comment_buffer:
    for n in comment_buffer:
        gs.add_node(n)
    
    count += len(comment_buffer)

print(f"== Total Comment nodes inserted: {count} ==")


== Insert Comment nodes ==
  committed 1000 Comment nodes
  committed 2000 Comment nodes
  committed 3000 Comment nodes
  committed 4000 Comment nodes
  committed 5000 Comment nodes
  committed 6000 Comment nodes
  committed 7000 Comment nodes
  committed 8000 Comment nodes
  committed 9000 Comment nodes
  committed 10000 Comment nodes
  committed 11000 Comment nodes
  committed 12000 Comment nodes
  committed 13000 Comment nodes
  committed 14000 Comment nodes
  committed 15000 Comment nodes
  committed 16000 Comment nodes
  committed 17000 Comment nodes
  committed 18000 Comment nodes
  committed 19000 Comment nodes
  committed 20000 Comment nodes
  committed 21000 Comment nodes
  committed 22000 Comment nodes
  committed 23000 Comment nodes
  committed 24000 Comment nodes
  committed 25000 Comment nodes
  committed 26000 Comment nodes
  committed 27000 Comment nodes
  committed 28000 Comment nodes
  committed 29000 Comment nodes
  committed 30000 Comment nodes
  committed 31000 Comm

In [12]:
# get tag_cache



# my paths

TAG_CSV = "data/tag_0_0.csv"


NODE_BATCH = 10



# here  A CACHE


tag_cache = {}       # key : id , val Node


# here  buffer

tag_buffer = []      

print("== Insert Tag nodes ==")

tag_df = pd.read_csv(
    TAG_CSV,
    sep="|",
    header=0,
    engine="python",
    quotechar='"',
    escapechar='\\',
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0
tag_buffer = []

for _, row in tag_df.iterrows():
    node = Node(
       labels=[Label("Tag")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("name", str, escape_cypher_string(row["name"])),
            Property("url", str, escape_cypher_string(row["url"]))
        ]
    )

    tag_cache[int(row["id"])] = node
    
print(f"== Total tag nodes inserted: {len(tag_cache)} ==")


== Insert Tag nodes ==
== Total tag nodes inserted: 16080 ==


In [12]:
# add tag nodes via HOGDB



# my paths

TAG_CSV = "data/tag_0_0.csv"


NODE_BATCH = 10



# INIT DB

db = Neo4jDatabase()
gs = GraphwithSubgraphStorage(db)


# here  A CACHE


tag_cache = {}       # key : id , val Node


# here  buffer

tag_buffer = []      

print("== Insert Tag nodes ==")

tag_df = pd.read_csv(
    TAG_CSV,
    sep="|",
    header=0,
    engine="python",
    quotechar='"',
    escapechar='\\',
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0
tag_buffer = []

for _, row in tag_df.iterrows():
    node = Node(
       labels=[Label("Tag")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("name", str, escape_cypher_string(row["name"])),
            Property("url", str, escape_cypher_string(row["url"]))
        ]
    )

    tag_cache[int(row["id"])] = node
    tag_buffer.append(node)

    # batch
    if len(tag_buffer) == NODE_BATCH:
        for n in tag_buffer:
            gs.add_node(n)

                     
        count += NODE_BATCH
        print(f"  committed {count} tag nodes")
        tag_buffer.clear()

# dernier batch
if tag_buffer:
    for n in tag_buffer:
        gs.add_node(n)
    
    count += len(tag_buffer)

print(f"== Total tag nodes inserted: {count} ==")


== Insert Tag nodes ==
  committed 1000 tag nodes
  committed 2000 tag nodes
  committed 3000 tag nodes
  committed 4000 tag nodes
  committed 5000 tag nodes
  committed 6000 tag nodes
  committed 7000 tag nodes
  committed 8000 tag nodes
  committed 9000 tag nodes
  committed 10000 tag nodes
  committed 11000 tag nodes
  committed 12000 tag nodes
  committed 13000 tag nodes
  committed 14000 tag nodes
  committed 15000 tag nodes
  committed 16000 tag nodes
== Total tag nodes inserted: 16080 ==


In [13]:
# cache Place nodes via HOGDB

PLACE_CSV = "data/place_0_0.csv"

print("== Insert Place nodes ==")

place_cache = {}
place_buffer = []

place_df = pd.read_csv(
    PLACE_CSV,
    sep="|",
    header=0,
    engine="python",
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in place_df.iterrows():
    node = Node(
        labels=[Label("Place")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("name", str, escape_cypher_string(row["name"])),
            Property("url", str, escape_cypher_string(row["url"])),
            Property("type", str, row["type"])
        ]
    )

    place_cache[int(row["id"])] = node
    

print(f"== Total Place nodes inserted: {count} ==")


== Insert Place nodes ==
== Total Place nodes inserted: 0 ==


In [15]:
# add Place nodes via HOGDB

PLACE_CSV = "data/place_0_0.csv"

print("== Insert Place nodes ==")

place_cache = {}
place_buffer = []

place_df = pd.read_csv(
    PLACE_CSV,
    sep="|",
    header=0,
    engine="python",
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in place_df.iterrows():
    node = Node(
        labels=[Label("Place")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("name", str, escape_cypher_string(row["name"])),
            Property("url", str, escape_cypher_string(row["url"])),
            Property("type", str, row["type"])
        ]
    )

    place_cache[int(row["id"])] = node
    place_buffer.append(node)

    if len(place_buffer) == NODE_BATCH:
        for n in place_buffer:
            gs.add_node(n)

        count += NODE_BATCH
        print(f"  committed {count} Place nodes")
        place_buffer.clear()

# last batch
if place_buffer:
    for n in place_buffer:
        gs.add_node(n)

    count += len(place_buffer)

print(f"== Total Place nodes inserted: {count} ==")


== Insert Place nodes ==
  committed 1000 Place nodes
== Total Place nodes inserted: 1460 ==


In [10]:
# cache Forum nodes via HOGDB

FORUM_CSV = "data/forum_0_0.csv"



forum_cache = {}
forum_buffer = []

forum_df = pd.read_csv(
    FORUM_CSV,
    sep="|",
    header=0,
    engine="python",
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in forum_df.iterrows():
    node = Node(
        labels=[Label("Forum")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("title", str, escape_cypher_string(row["title"])),
            Property("creationDate", int, int(row["creationDate"]))
        ]
    )

    forum_cache[int(row["id"])] = node
    
print(f"== Total forum cache: {count} ==")


== Insert Forum nodes ==
== Total Forum nodes inserted: 0 ==


In [16]:
# add Forum nodes via HOGDB

FORUM_CSV = "data/forum_0_0.csv"

print("== Insert Forum nodes ==")

forum_cache = {}
forum_buffer = []

forum_df = pd.read_csv(
    FORUM_CSV,
    sep="|",
    header=0,
    engine="python",
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in forum_df.iterrows():
    node = Node(
        labels=[Label("Forum")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("title", str, escape_cypher_string(row["title"])),
            Property("creationDate", int, int(row["creationDate"]))
        ]
    )

    forum_cache[int(row["id"])] = node
    forum_buffer.append(node)

    if len(forum_buffer) == NODE_BATCH:
        for n in forum_buffer:
            gs.add_node(n)

        count += NODE_BATCH
        print(f"  committed {count} Forum nodes")
        forum_buffer.clear()

# last batch
if forum_buffer:
    for n in forum_buffer:
        gs.add_node(n)

    count += len(forum_buffer)

print(f"== Total Forum nodes inserted: {count} ==")


== Insert Forum nodes ==
  committed 1000 Forum nodes
  committed 2000 Forum nodes
  committed 3000 Forum nodes
  committed 4000 Forum nodes
  committed 5000 Forum nodes
  committed 6000 Forum nodes
  committed 7000 Forum nodes
  committed 8000 Forum nodes
  committed 9000 Forum nodes
  committed 10000 Forum nodes
  committed 11000 Forum nodes
  committed 12000 Forum nodes
  committed 13000 Forum nodes
  committed 14000 Forum nodes
  committed 15000 Forum nodes
  committed 16000 Forum nodes
  committed 17000 Forum nodes
  committed 18000 Forum nodes
  committed 19000 Forum nodes
  committed 20000 Forum nodes
  committed 21000 Forum nodes
  committed 22000 Forum nodes
  committed 23000 Forum nodes
  committed 24000 Forum nodes
  committed 25000 Forum nodes
  committed 26000 Forum nodes
  committed 27000 Forum nodes
  committed 28000 Forum nodes
  committed 29000 Forum nodes
  committed 30000 Forum nodes
  committed 31000 Forum nodes
== Total Forum nodes inserted: 31097 ==


In [11]:
# cache Organisation nodes via HOGDB

ORG_CSV = "data/organisation_0_0.csv"



org_cache = {}
org_buffer = []

org_df = pd.read_csv(
    ORG_CSV,
    sep="|",
    header=0,
    engine="python",
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in org_df.iterrows():
    node = Node(
        labels=[Label("Organisation")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("type", str, row["type"]),
            Property("name", str, escape_cypher_string(row["name"])),
            Property("url", str, escape_cypher_string(row["url"]))
        ]
    )

    org_cache[int(row["id"])] = node
    

print(f"== Total Organisation nodes inserted: {count} ==")


== Insert Organisation nodes ==
== Total Organisation nodes inserted: 0 ==


In [17]:
# add Organisation nodes via HOGDB

ORG_CSV = "data/organisation_0_0.csv"

print("== Insert Organisation nodes ==")

org_cache = {}
org_buffer = []

org_df = pd.read_csv(
    ORG_CSV,
    sep="|",
    header=0,
    engine="python",
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in org_df.iterrows():
    node = Node(
        labels=[Label("Organisation")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("type", str, row["type"]),
            Property("name", str, escape_cypher_string(row["name"])),
            Property("url", str, escape_cypher_string(row["url"]))
        ]
    )

    org_cache[int(row["id"])] = node
    org_buffer.append(node)

    if len(org_buffer) == NODE_BATCH:
        for n in org_buffer:
            gs.add_node(n)

        count += NODE_BATCH
        print(f"  committed {count} Organisation nodes")
        org_buffer.clear()

# last batch
if org_buffer:
    for n in org_buffer:
        gs.add_node(n)

    count += len(org_buffer)

print(f"== Total Organisation nodes inserted: {count} ==")


== Insert Organisation nodes ==
  committed 1000 Organisation nodes
  committed 2000 Organisation nodes
  committed 3000 Organisation nodes
  committed 4000 Organisation nodes
  committed 5000 Organisation nodes
  committed 6000 Organisation nodes
  committed 7000 Organisation nodes
== Total Organisation nodes inserted: 7955 ==


In [14]:
# cache TagClass nodes via HOGDB

TAGCLASS_CSV = "data/tagclass_0_0.csv"


tagclass_cache = {}
tagclass_buffer = []

tagclass_df = pd.read_csv(
    TAGCLASS_CSV,
    sep="|",
    header=0,
    engine="python",
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in tagclass_df.iterrows():
    node = Node(
        labels=[Label("TagClass")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("name", str, escape_cypher_string(row["name"])),
            Property("url", str, escape_cypher_string(row["url"]))
        ]
    )

    tagclass_cache[int(row["id"])] = node
    
print(f"== Total TagClass nodes inserted: {count} ==")


== Total TagClass nodes inserted: 0 ==


In [18]:
# add TagClass nodes via HOGDB

TAGCLASS_CSV = "data/tagclass_0_0.csv"

print("== Insert TagClass nodes ==")

tagclass_cache = {}
tagclass_buffer = []

tagclass_df = pd.read_csv(
    TAGCLASS_CSV,
    sep="|",
    header=0,
    engine="python",
    encoding="utf-8"
)

NODE_BATCH = 1000
count = 0

for _, row in tagclass_df.iterrows():
    node = Node(
        labels=[Label("TagClass")],
        properties=[
            Property("id", int, int(row["id"])),
            Property("name", str, escape_cypher_string(row["name"])),
            Property("url", str, escape_cypher_string(row["url"]))
        ]
    )

    tagclass_cache[int(row["id"])] = node
    tagclass_buffer.append(node)

    if len(tagclass_buffer) == NODE_BATCH:
        for n in tagclass_buffer:
            gs.add_node(n)

        count += NODE_BATCH
        print(f"  committed {count} TagClass nodes")
        tagclass_buffer.clear()

# last batch
if tagclass_buffer:
    for n in tagclass_buffer:
        gs.add_node(n)

    count += len(tagclass_buffer)

print(f"== Total TagClass nodes inserted: {count} ==")


== Insert TagClass nodes ==
== Total TagClass nodes inserted: 71 ==


In [ ]:
print("NOW WE ADD THE EDGES ")

In [ ]:
#HOGDB do not add indexes automatically, if you want to add indexes for edges please run the belo query in neo4j
CREATE CONSTRAINT IF NOT EXISTS FOR (e:_edge) REQUIRE e.id IS UNIQUE;

In [ ]:
#In order to run add_edge correctely, it's necessary to add id to edges, so please run this func to add id in the edges csv files 

# genere a new csv file with id column
import pandas as pd
import os

# dossier

DATA_DIR = "data/edges_sans_id"
DATA_ID_DIR ="data/edges"

# list of files +  id prefix
edge_files = {
    "comment_hasCreator_person_0_0.csv": "hasCreator",
    "comment_hasTag_tag_0_0.csv": "hasTag",
    "comment_isLocatedIn_place_0_0.csv": "isLocatedIn",
    "comment_replyOf_comment_0_0.csv": "replyOfComment",
    "comment_replyOf_post_0_0.csv": "replyOfPost",
    "forum_containerOf_post_0_0.csv": "containerOf",
    "forum_hasMember_person_0_0.csv": "hasMember",
    "forum_hasModerator_person_0_0.csv": "hasModerator",
    "forum_hasTag_tag_0_0.csv": "forumHasTag",
    "person_hasInterest_tag_0_0.csv": "hasInterest",
    "person_isLocatedIn_place_0_0.csv": "personLocatedIn",
    "person_knows_person_0_0.csv": "knows",
    "person_likes_comment_0_0.csv": "likesComment",
    "person_likes_post_0_0.csv": "likesPost",
    "person_studyAt_organisation_0_0.csv": "studyAt",
    "person_workAt_organisation_0_0.csv": "workAt",
    "post_hasCreator_person_0_0.csv": "postCreator",
    "post_hasTag_tag_0_0.csv": "postHasTag",
    "post_isLocatedIn_place_0_0.csv": "postLocatedIn",
    "organisation_isLocatedIn_place_0_0.csv": "orgLocatedIn",
    "place_isPartOf_place_0_0.csv": "isPartOf",
    "tag_hasType_tagclass_0_0.csv": "hasType",
    "tagclass_isSubclassOf_tagclass_0_0.csv": "subclassOf"
}

print("== Add ID to all edge CSV files ==")

for file_name, prefix in edge_files.items():
    src = os.path.join(DATA_DIR, file_name)
    dst = os.path.join(DATA_ID_DIR, file_name.replace(".csv", "_with_id.csv"))

    print(f"Processing {file_name}...")

    df = pd.read_csv(src, sep="|", engine="python", encoding="utf-8")

    #create a  unique id
    df["id"] = [f"{prefix}{i+1}" for i in range(len(df))]

    # add id at the first column
    cols = ["id"] + [c for c in df.columns if c != "id"]
    df = df[cols]

    # save
    df.to_csv(dst, sep="|", index=False, encoding="utf-8")

    print(f"   -> Wrote {dst} ({len(df)} rows)")

print("== DONE =====")


== Add ID to all edge CSV files ==
Processing comment_hasCreator_person_0_0.csv...
   -> Wrote data/edges\comment_hasCreator_person_0_0_with_id.csv (523222 rows)
Processing comment_hasTag_tag_0_0.csv...
   -> Wrote data/edges\comment_hasTag_tag_0_0_with_id.csv (680738 rows)
Processing comment_isLocatedIn_place_0_0.csv...
   -> Wrote data/edges\comment_isLocatedIn_place_0_0_with_id.csv (523222 rows)
Processing comment_replyOf_comment_0_0.csv...
   -> Wrote data/edges\comment_replyOf_comment_0_0_with_id.csv (265931 rows)
Processing comment_replyOf_post_0_0.csv...
   -> Wrote data/edges\comment_replyOf_post_0_0_with_id.csv (257291 rows)
Processing forum_containerOf_post_0_0.csv...
   -> Wrote data/edges\forum_containerOf_post_0_0_with_id.csv (324825 rows)
Processing forum_hasMember_person_0_0.csv...
   -> Wrote data/edges\forum_hasMember_person_0_0_with_id.csv (404952 rows)
Processing forum_hasModerator_person_0_0.csv...
   -> Wrote data/edges\forum_hasModerator_person_0_0_with_id.csv (31

In [ ]:
#generic func to add all type of edges in the database

import pandas as pd

EDGE_BATCH = 1000

def insert_edges(csv_path, source_col, target_col,
                 source_cache, target_cache,
                 label_name, extra_props=[]):

    print(f"== Insert {label_name} edges ==")

    edge_df = pd.read_csv(
        csv_path,
        sep="|",
        header=0,
        engine="python",
        encoding="utf-8"
    )

    edge_buffer = []
    count = 0

    for _, row in edge_df.iterrows():
        s_id = int(row[source_col])
        t_id = int(row[target_col])

        # vérifier existence des nodes
        if s_id not in source_cache or t_id not in target_cache:
            continue

        # propriétés de base
        props = [Property("id", str, row["id"])]

        # propriétés supplémentaires
        for prop in extra_props:
            props.append(Property(prop, str, row[prop]))

        e = Edge(
            source_cache[s_id],
            target_cache[t_id],
            Label(label_name),
            props
        )

        edge_buffer.append(e)

        # batch insert
        if len(edge_buffer) == EDGE_BATCH:
            for edge in edge_buffer:
                gs.add_edge(edge)

            count += EDGE_BATCH
            print(f"  committed {count} {label_name}")
            edge_buffer.clear()

    # dernier batch
    if edge_buffer:
        for edge in edge_buffer:
            gs.add_edge(edge)

        count += len(edge_buffer)

    print(f"== Total {label_name}: {count} ==")


In [ ]:
# func to get  HASTAG cache_edge (needed for subgraph creation)

import pandas as pd

EDGE_BATCH = 1000

def insert_ht_cache_edges(csv_path, source_col, target_col,
                 source_cache, target_cache,
                 label_name, extra_props=[]):

    print(f"== Insert {label_name} edges ==")

    edge_df = pd.read_csv(
        csv_path,
        sep="|",
        header=0,
        engine="python",
        encoding="utf-8"
    )
    edge_cache = {}      # Key : (c_id, t_id) , val : Edge
    edge_buffer = []
    count = 0

    for _, row in edge_df.iterrows():
        s_id = int(row[source_col])
        t_id = int(row[target_col])

        # vérifier existence des nodes
        if s_id not in source_cache or t_id not in target_cache:
            continue

        # propriétés de base
        props = [Property("id", str, row["id"])]

        # propriétés supplémentaires
        for prop in extra_props:
            props.append(Property(prop, str, row[prop]))

        e = Edge(
            source_cache[s_id],
            target_cache[t_id],
            Label(label_name),
            props
        )
        edge_cache[s_id, t_id] = e
   

        
    count = len(edge_cache)

    print(f"== Total : {count} ==")
    return edge_cache

In [17]:
edge_cache= insert_ht_cache_edges(
    "data/edges/comment_hasTag_tag_0_0_with_id.csv",
    "Comment.id", "Tag.id",
    comment_cache, tag_cache,
    "HAS_TAG"
)
len(edge_cache)

== Insert HAS_TAG edges ==
== Total : 680738 ==


680738

In [ ]:
#COMMENT HAS_TAG TAG
insert_edges(
    "data/edges/comment_hasTag_tag_0_0_with_id.csv",
    "Comment.id", "Tag.id",
    comment_cache, tag_cache,
    "HAS_TAG"
)
print("done")


== Insert HAS_TAG edges ==
  committed 500 HAS_TAG
  committed 1000 HAS_TAG
  committed 1500 HAS_TAG
  committed 2000 HAS_TAG
  committed 2500 HAS_TAG
  committed 3000 HAS_TAG
  committed 3500 HAS_TAG
  committed 4000 HAS_TAG
  committed 4500 HAS_TAG
  committed 5000 HAS_TAG
  committed 5500 HAS_TAG
  committed 6000 HAS_TAG
  committed 6500 HAS_TAG
  committed 7000 HAS_TAG
  committed 7500 HAS_TAG
  committed 8000 HAS_TAG
  committed 8500 HAS_TAG
  committed 9000 HAS_TAG
  committed 9500 HAS_TAG
  committed 10000 HAS_TAG
  committed 10500 HAS_TAG
  committed 11000 HAS_TAG
  committed 11500 HAS_TAG
  committed 12000 HAS_TAG
  committed 12500 HAS_TAG
  committed 13000 HAS_TAG
  committed 13500 HAS_TAG
  committed 14000 HAS_TAG
  committed 14500 HAS_TAG
  committed 15000 HAS_TAG
  committed 15500 HAS_TAG
  committed 16000 HAS_TAG
  committed 16500 HAS_TAG
  committed 17000 HAS_TAG
  committed 17500 HAS_TAG
  committed 18000 HAS_TAG
  committed 18500 HAS_TAG
  committed 19000 HAS_TAG
  com

In [ ]:
# COMMENT HAS_CREATOR PERSON 
insert_edges(
    "data/edges/comment_hasCreator_person_0_0_with_id.csv",
    "Comment.id", "Person.id",
    comment_cache, person_cache,
    "HAS_CREATOR"
)
print("done")

== Insert HAS_CREATOR edges ==
  committed 1000 HAS_CREATOR
  committed 2000 HAS_CREATOR
  committed 3000 HAS_CREATOR
  committed 4000 HAS_CREATOR
  committed 5000 HAS_CREATOR
  committed 6000 HAS_CREATOR
  committed 7000 HAS_CREATOR
  committed 8000 HAS_CREATOR
  committed 9000 HAS_CREATOR
  committed 10000 HAS_CREATOR
  committed 11000 HAS_CREATOR
  committed 12000 HAS_CREATOR
  committed 13000 HAS_CREATOR
  committed 14000 HAS_CREATOR
  committed 15000 HAS_CREATOR
  committed 16000 HAS_CREATOR
  committed 17000 HAS_CREATOR
  committed 18000 HAS_CREATOR
  committed 19000 HAS_CREATOR
  committed 20000 HAS_CREATOR
  committed 21000 HAS_CREATOR
  committed 22000 HAS_CREATOR
  committed 23000 HAS_CREATOR
  committed 24000 HAS_CREATOR
  committed 25000 HAS_CREATOR
  committed 26000 HAS_CREATOR
  committed 27000 HAS_CREATOR
  committed 28000 HAS_CREATOR
  committed 29000 HAS_CREATOR
  committed 30000 HAS_CREATOR
  committed 31000 HAS_CREATOR
  committed 32000 HAS_CREATOR
  committed 33000 

In [ ]:
#COMMENT IS_LOCATED_IN PLACE
insert_edges(
    "data/edges/comment_isLocatedIn_place_0_0_with_id.csv",
    "Comment.id", "Place.id",
    comment_cache, place_cache,
    "IS_LOCATED_IN"
)
print("done")

== Insert IS_LOCATED_IN edges ==
  committed 1000 IS_LOCATED_IN
  committed 2000 IS_LOCATED_IN
  committed 3000 IS_LOCATED_IN
  committed 4000 IS_LOCATED_IN
  committed 5000 IS_LOCATED_IN
  committed 6000 IS_LOCATED_IN
  committed 7000 IS_LOCATED_IN
  committed 8000 IS_LOCATED_IN
  committed 9000 IS_LOCATED_IN
  committed 10000 IS_LOCATED_IN
  committed 11000 IS_LOCATED_IN
  committed 12000 IS_LOCATED_IN
  committed 13000 IS_LOCATED_IN
  committed 14000 IS_LOCATED_IN
  committed 15000 IS_LOCATED_IN
  committed 16000 IS_LOCATED_IN
  committed 17000 IS_LOCATED_IN
  committed 18000 IS_LOCATED_IN
  committed 19000 IS_LOCATED_IN
  committed 20000 IS_LOCATED_IN
  committed 21000 IS_LOCATED_IN
  committed 22000 IS_LOCATED_IN
  committed 23000 IS_LOCATED_IN
  committed 24000 IS_LOCATED_IN
  committed 25000 IS_LOCATED_IN
  committed 26000 IS_LOCATED_IN
  committed 27000 IS_LOCATED_IN
  committed 28000 IS_LOCATED_IN
  committed 29000 IS_LOCATED_IN
  committed 30000 IS_LOCATED_IN
  committed 3100

In [30]:
#COMMENT REPLY OF COMMENT
insert_edges(
    "data/edges/comment_replyOf_comment_0_0_with_id.csv",
    "Comment1.id", "Comment2.id",
    comment_cache, comment_cache,
    "REPLY_OF"
)
print("done")

== Insert REPLY_OF edges ==
  committed 1000 REPLY_OF
  committed 2000 REPLY_OF
  committed 3000 REPLY_OF
  committed 4000 REPLY_OF
  committed 5000 REPLY_OF
  committed 6000 REPLY_OF
  committed 7000 REPLY_OF
  committed 8000 REPLY_OF
  committed 9000 REPLY_OF
  committed 10000 REPLY_OF
  committed 11000 REPLY_OF
  committed 12000 REPLY_OF
  committed 13000 REPLY_OF
  committed 14000 REPLY_OF
  committed 15000 REPLY_OF
  committed 16000 REPLY_OF
  committed 17000 REPLY_OF
  committed 18000 REPLY_OF
  committed 19000 REPLY_OF
  committed 20000 REPLY_OF
  committed 21000 REPLY_OF
  committed 22000 REPLY_OF
  committed 23000 REPLY_OF
  committed 24000 REPLY_OF
  committed 25000 REPLY_OF
  committed 26000 REPLY_OF
  committed 27000 REPLY_OF
  committed 28000 REPLY_OF
  committed 29000 REPLY_OF
  committed 30000 REPLY_OF
  committed 31000 REPLY_OF
  committed 32000 REPLY_OF
  committed 33000 REPLY_OF
  committed 34000 REPLY_OF
  committed 35000 REPLY_OF
  committed 36000 REPLY_OF
  committ

In [31]:
#COMMENT REPLYOF POST

insert_edges(
    "data/edges/comment_replyOf_post_0_0_with_id.csv",
    "Comment.id", "Post.id",
    comment_cache, post_cache,
    "REPLY_OF"
)
print("done")

== Insert REPLY_OF edges ==
  committed 1000 REPLY_OF
  committed 2000 REPLY_OF
  committed 3000 REPLY_OF
  committed 4000 REPLY_OF
  committed 5000 REPLY_OF
  committed 6000 REPLY_OF
  committed 7000 REPLY_OF
  committed 8000 REPLY_OF
  committed 9000 REPLY_OF
  committed 10000 REPLY_OF
  committed 11000 REPLY_OF
  committed 12000 REPLY_OF
  committed 13000 REPLY_OF
  committed 14000 REPLY_OF
  committed 15000 REPLY_OF
  committed 16000 REPLY_OF
  committed 17000 REPLY_OF
  committed 18000 REPLY_OF
  committed 19000 REPLY_OF
  committed 20000 REPLY_OF
  committed 21000 REPLY_OF
  committed 22000 REPLY_OF
  committed 23000 REPLY_OF
  committed 24000 REPLY_OF
  committed 25000 REPLY_OF
  committed 26000 REPLY_OF
  committed 27000 REPLY_OF
  committed 28000 REPLY_OF
  committed 29000 REPLY_OF
  committed 30000 REPLY_OF
  committed 31000 REPLY_OF
  committed 32000 REPLY_OF
  committed 33000 REPLY_OF
  committed 34000 REPLY_OF
  committed 35000 REPLY_OF
  committed 36000 REPLY_OF
  committ

In [ ]:
#FORUM CONTAINTER_OF POST
insert_edges(
    "data/edges/forum_containerOf_post_0_0_with_id.csv",
    "Forum.id", "Post.id",
    forum_cache, post_cache,
    "CONTAINER_OF"
)


== Insert CONTAINER_OF edges ==
  committed 1000 CONTAINER_OF
  committed 2000 CONTAINER_OF
  committed 3000 CONTAINER_OF
  committed 4000 CONTAINER_OF
  committed 5000 CONTAINER_OF
  committed 6000 CONTAINER_OF
  committed 7000 CONTAINER_OF
  committed 8000 CONTAINER_OF
  committed 9000 CONTAINER_OF
  committed 10000 CONTAINER_OF
  committed 11000 CONTAINER_OF
  committed 12000 CONTAINER_OF
  committed 13000 CONTAINER_OF
  committed 14000 CONTAINER_OF
  committed 15000 CONTAINER_OF
  committed 16000 CONTAINER_OF
  committed 17000 CONTAINER_OF
  committed 18000 CONTAINER_OF
  committed 19000 CONTAINER_OF
  committed 20000 CONTAINER_OF
  committed 21000 CONTAINER_OF
  committed 22000 CONTAINER_OF
  committed 23000 CONTAINER_OF
  committed 24000 CONTAINER_OF
  committed 25000 CONTAINER_OF
  committed 26000 CONTAINER_OF
  committed 27000 CONTAINER_OF
  committed 28000 CONTAINER_OF
  committed 29000 CONTAINER_OF
  committed 30000 CONTAINER_OF
  committed 31000 CONTAINER_OF
  committed 3200

In [ ]:
#FORUM HAS_MEMBER PERSON
insert_edges(
    "data/edges/forum_hasMember_person_0_0_with_id.csv",
    "Forum.id", "Person.id",
    forum_cache, person_cache,
    "HAS_MEMBER",
    extra_props=["joinDate"]
)


== Insert HAS_MEMBER edges ==
  committed 1000 HAS_MEMBER
  committed 2000 HAS_MEMBER
  committed 3000 HAS_MEMBER
  committed 4000 HAS_MEMBER
  committed 5000 HAS_MEMBER
  committed 6000 HAS_MEMBER
  committed 7000 HAS_MEMBER
  committed 8000 HAS_MEMBER
  committed 9000 HAS_MEMBER
  committed 10000 HAS_MEMBER
  committed 11000 HAS_MEMBER
  committed 12000 HAS_MEMBER
  committed 13000 HAS_MEMBER
  committed 14000 HAS_MEMBER
  committed 15000 HAS_MEMBER
  committed 16000 HAS_MEMBER
  committed 17000 HAS_MEMBER
  committed 18000 HAS_MEMBER
  committed 19000 HAS_MEMBER
  committed 20000 HAS_MEMBER
  committed 21000 HAS_MEMBER
  committed 22000 HAS_MEMBER
  committed 23000 HAS_MEMBER
  committed 24000 HAS_MEMBER
  committed 25000 HAS_MEMBER
  committed 26000 HAS_MEMBER
  committed 27000 HAS_MEMBER
  committed 28000 HAS_MEMBER
  committed 29000 HAS_MEMBER
  committed 30000 HAS_MEMBER
  committed 31000 HAS_MEMBER
  committed 32000 HAS_MEMBER
  committed 33000 HAS_MEMBER
  committed 34000 HAS_

In [ ]:
# FORUM HAS_MODERATOR PERSON
insert_edges(
    "data/edges/forum_hasModerator_person_0_0_with_id.csv",
    "Forum.id", "Person.id",
    forum_cache, person_cache,
    "HAS_MODERATOR"
)
print("done")

== Insert HAS_MODERATOR edges ==
  committed 1000 HAS_MODERATOR
  committed 2000 HAS_MODERATOR
  committed 3000 HAS_MODERATOR
  committed 4000 HAS_MODERATOR
  committed 5000 HAS_MODERATOR
  committed 6000 HAS_MODERATOR
  committed 7000 HAS_MODERATOR
  committed 8000 HAS_MODERATOR
  committed 9000 HAS_MODERATOR
  committed 10000 HAS_MODERATOR
  committed 11000 HAS_MODERATOR
  committed 12000 HAS_MODERATOR
  committed 13000 HAS_MODERATOR
  committed 14000 HAS_MODERATOR
  committed 15000 HAS_MODERATOR
  committed 16000 HAS_MODERATOR
  committed 17000 HAS_MODERATOR
  committed 18000 HAS_MODERATOR
  committed 19000 HAS_MODERATOR
  committed 20000 HAS_MODERATOR
  committed 21000 HAS_MODERATOR
  committed 22000 HAS_MODERATOR
  committed 23000 HAS_MODERATOR
  committed 24000 HAS_MODERATOR
  committed 25000 HAS_MODERATOR
  committed 26000 HAS_MODERATOR
  committed 27000 HAS_MODERATOR
  committed 28000 HAS_MODERATOR
  committed 29000 HAS_MODERATOR
  committed 30000 HAS_MODERATOR
  committed 3100

In [ ]:
#PERSON KNOWS PERSON
insert_edges(
    "data/edges/person_knows_person_0_0_with_id.csv",
    "Person1.id", "Person2.id",
    person_cache, person_cache,
    "KNOWS",
    extra_props=["creationDate"]
)


== Insert KNOWS edges ==
  committed 1000 KNOWS
  committed 2000 KNOWS
  committed 3000 KNOWS
  committed 4000 KNOWS
  committed 5000 KNOWS
  committed 6000 KNOWS
  committed 7000 KNOWS
  committed 8000 KNOWS
  committed 9000 KNOWS
  committed 10000 KNOWS
  committed 11000 KNOWS
  committed 12000 KNOWS
  committed 13000 KNOWS
  committed 14000 KNOWS
  committed 15000 KNOWS
  committed 16000 KNOWS
  committed 17000 KNOWS
  committed 18000 KNOWS
  committed 19000 KNOWS
  committed 20000 KNOWS
  committed 21000 KNOWS
  committed 22000 KNOWS
  committed 23000 KNOWS
  committed 24000 KNOWS
  committed 25000 KNOWS
  committed 26000 KNOWS
  committed 27000 KNOWS
  committed 28000 KNOWS
  committed 29000 KNOWS
  committed 30000 KNOWS
  committed 31000 KNOWS
  committed 32000 KNOWS
  committed 33000 KNOWS
  committed 34000 KNOWS
  committed 35000 KNOWS
  committed 36000 KNOWS
  committed 37000 KNOWS
  committed 38000 KNOWS
  committed 39000 KNOWS
  committed 40000 KNOWS
  committed 41000 KNOWS


In [ ]:
# PERSON LIKES COMMENT
insert_edges(
    "data/edges/person_likes_comment_0_0_with_id.csv",
    "Person.id", "Comment.id",
    person_cache, comment_cache,
    "LIKES",
    extra_props=["creationDate"]
)


== Insert LIKES edges ==
  committed 1000 LIKES


In [ ]:
#PERSON LIKES POST
insert_edges(
    "data/edges/person_likes_post_0_0_with_id.csv",
    "Person.id", "Post.id",
    person_cache, post_cache,
    "LIKES",
    extra_props=["creationDate"]
)


In [28]:
#STUDY_ATabs

insert_edges(
    "data/edges/person_studyAt_organisation_0_0_with_id.csv",
    "Person.id", "Organisation.id",
    person_cache, org_cache,
    "STUDY_AT",
    extra_props=["classYear"]
)



== Insert STUDY_AT edges ==
  committed 500 STUDY_AT
  committed 1000 STUDY_AT
== Total STUDY_AT: 1209 ==


In [29]:
#WORKS AT
insert_edges(
    "data/edges/person_workAt_organisation_0_0_with_id.csv",
    "Person.id", "Organisation.id",
    person_cache, org_cache,
    "WORK_AT",
    extra_props=["workFrom"]
)

== Insert WORK_AT edges ==
  committed 500 WORK_AT
  committed 1000 WORK_AT
  committed 1500 WORK_AT
  committed 2000 WORK_AT
  committed 2500 WORK_AT
  committed 3000 WORK_AT
== Total WORK_AT: 3313 ==


In [ ]:
#POST HAS_CREATOR PERSON
insert_edges(
    "data/edges/post_hasCreator_person_0_0_with_id.csv",
    "Post.id", "Person.id",
    post_cache, person_cache,
    "HAS_CREATOR"
)
print ("done")


In [18]:
#POST HAS_TAG TAG
insert_edges(
    "data/edges/post_hasTag_tag_0_0_with_id.csv",
    "Post.id", "Tag.id",
    post_cache, tag_cache,
    "HAS_TAG"
)


== Insert HAS_TAG edges ==
  committed 1000 HAS_TAG
  committed 2000 HAS_TAG
  committed 3000 HAS_TAG
  committed 4000 HAS_TAG
  committed 5000 HAS_TAG
  committed 6000 HAS_TAG
  committed 7000 HAS_TAG
  committed 8000 HAS_TAG
  committed 9000 HAS_TAG
  committed 10000 HAS_TAG
  committed 11000 HAS_TAG
  committed 12000 HAS_TAG
  committed 13000 HAS_TAG
  committed 14000 HAS_TAG
  committed 15000 HAS_TAG
  committed 16000 HAS_TAG
  committed 17000 HAS_TAG
  committed 18000 HAS_TAG
  committed 19000 HAS_TAG
  committed 20000 HAS_TAG
  committed 21000 HAS_TAG
  committed 22000 HAS_TAG
  committed 23000 HAS_TAG
  committed 24000 HAS_TAG
  committed 25000 HAS_TAG
  committed 26000 HAS_TAG
  committed 27000 HAS_TAG
  committed 28000 HAS_TAG
  committed 29000 HAS_TAG
  committed 30000 HAS_TAG
  committed 31000 HAS_TAG
  committed 32000 HAS_TAG
  committed 33000 HAS_TAG
  committed 34000 HAS_TAG
  committed 35000 HAS_TAG
  committed 36000 HAS_TAG
  committed 37000 HAS_TAG
  committed 38000 HA

In [19]:
#POST IS_LOCATED_IN PLACE
insert_edges(
    "data/edges/post_isLocatedIn_place_0_0_with_id.csv",
    "Post.id", "Place.id",
    post_cache, place_cache,
    "IS_LOCATED_IN"
)


== Insert IS_LOCATED_IN edges ==
  committed 1000 IS_LOCATED_IN
  committed 2000 IS_LOCATED_IN
  committed 3000 IS_LOCATED_IN
  committed 4000 IS_LOCATED_IN
  committed 5000 IS_LOCATED_IN
  committed 6000 IS_LOCATED_IN
  committed 7000 IS_LOCATED_IN
  committed 8000 IS_LOCATED_IN
  committed 9000 IS_LOCATED_IN
  committed 10000 IS_LOCATED_IN
  committed 11000 IS_LOCATED_IN
  committed 12000 IS_LOCATED_IN
  committed 13000 IS_LOCATED_IN
  committed 14000 IS_LOCATED_IN
  committed 15000 IS_LOCATED_IN
  committed 16000 IS_LOCATED_IN
  committed 17000 IS_LOCATED_IN
  committed 18000 IS_LOCATED_IN
  committed 19000 IS_LOCATED_IN
  committed 20000 IS_LOCATED_IN
  committed 21000 IS_LOCATED_IN
  committed 22000 IS_LOCATED_IN
  committed 23000 IS_LOCATED_IN
  committed 24000 IS_LOCATED_IN
  committed 25000 IS_LOCATED_IN
  committed 26000 IS_LOCATED_IN
  committed 27000 IS_LOCATED_IN
  committed 28000 IS_LOCATED_IN
  committed 29000 IS_LOCATED_IN
  committed 30000 IS_LOCATED_IN
  committed 3100

In [20]:
#PLACE IS_PART_OF PPLACE
insert_edges(
    "data/edges/place_isPartOf_place_0_0_with_id.csv",
    "Place1.id", "Place2.id",
    place_cache, place_cache,
    "IS_PART_OF"
)




== Insert IS_PART_OF edges ==
  committed 1000 IS_PART_OF
== Total IS_PART_OF: 1454 ==


In [21]:
#TAG HAS_TYPE TTAGCLASS
insert_edges(
    "data/edges/tag_hasType_tagclass_0_0_with_id.csv",
    "Tag.id", "TagClass.id",
    tag_cache, tagclass_cache,
    "HAS_TYPE"
)


== Insert HAS_TYPE edges ==
  committed 1000 HAS_TYPE
  committed 2000 HAS_TYPE
  committed 3000 HAS_TYPE
  committed 4000 HAS_TYPE
  committed 5000 HAS_TYPE
  committed 6000 HAS_TYPE
  committed 7000 HAS_TYPE
  committed 8000 HAS_TYPE
  committed 9000 HAS_TYPE
  committed 10000 HAS_TYPE
  committed 11000 HAS_TYPE
  committed 12000 HAS_TYPE
  committed 13000 HAS_TYPE
  committed 14000 HAS_TYPE
  committed 15000 HAS_TYPE
  committed 16000 HAS_TYPE
== Total HAS_TYPE: 16080 ==


In [22]:
# tagclass IS_SUBCLASS_OF tagclass
insert_edges(
    "data/edges/tagclass_isSubclassOf_tagclass_0_0_with_id.csv",
    "TagClass1.id", "TagClass2.id",
    tagclass_cache, tagclass_cache,
    "IS_SUBCLASS_OF"
)

== Insert IS_SUBCLASS_OF edges ==
== Total IS_SUBCLASS_OF: 70 ==


In [ ]:
# Plan B in case of interrumption during the process of taggedComment insertion in the DB : 
# Fist, you run this query in noe4j browser to get the number of HAS_TAG nodes connected to comment and tag nodes via _adjacency() [ "MATCH (c:Comment)-[:_adjacency]->(ht:HAS_TAG)-[:_adjacency]->(t:Tag)RETURN count(ht) AS nombre_HAS_TAG;"]
# Second, update the  start_point that will be the value returned by the query since the dict index starts with 0
# Third, you run this funct to slice the edge_cache  and get edge_cache_copy containig only the remaining hasTag edges
# Finally, rerun the below cell (#add taggedComment Subgraphs) 
from itertools import islice
start_point = 153026
edge_cache_copy = dict(islice(edge_cache.items(), start_point, None))


In [ ]:

#add taggedComment Subgraphs

SUBGRAPH_BATCH = 1000

print("===Insert taggedComment subgraphs ")

buffer = []
count = 0

for (s_id, t_id), e in edge_cache.items(): # replace edge_cache by edge_cache_copy in case of interruption (see the above cell)
    sg = Subgraph(
        subgraph_nodes=[
            comment_cache[s_id],
            tag_cache[t_id]
        ],
        subgraph_edges=[e],
        labels=[Label("taggedComment")],
        properties=[
            Property("id", str, f"tc_{s_id}_{t_id}")
        ]
    )

    buffer.append(sg)

    if len(buffer) == SUBGRAPH_BATCH:
        for s in buffer:
            gs.add_subgraph(s)
        count += len(buffer)
        buffer.clear()
        print(f"  inserted {count} subgraphs")

# flush last batch
if buffer:
    for s in buffer:
        gs.add_subgraph(s)
    count += len(buffer)

print(f"== DONE: {count} subgraphs inserted ==")





===Insert taggedComment subgraphs 
  inserted 1000 subgraphs
  inserted 2000 subgraphs
  inserted 3000 subgraphs
  inserted 4000 subgraphs
  inserted 5000 subgraphs
  inserted 6000 subgraphs
  inserted 7000 subgraphs
  inserted 8000 subgraphs
  inserted 9000 subgraphs
  inserted 10000 subgraphs
  inserted 11000 subgraphs
  inserted 12000 subgraphs
  inserted 13000 subgraphs
  inserted 14000 subgraphs
  inserted 15000 subgraphs
  inserted 16000 subgraphs
  inserted 17000 subgraphs
  inserted 18000 subgraphs
  inserted 19000 subgraphs
  inserted 20000 subgraphs
  inserted 21000 subgraphs
  inserted 22000 subgraphs
  inserted 23000 subgraphs
  inserted 24000 subgraphs
  inserted 25000 subgraphs
  inserted 26000 subgraphs
  inserted 27000 subgraphs
  inserted 28000 subgraphs
  inserted 29000 subgraphs
  inserted 30000 subgraphs
  inserted 31000 subgraphs
  inserted 32000 subgraphs
  inserted 33000 subgraphs
  inserted 34000 subgraphs
  inserted 35000 subgraphs
  inserted 36000 subgraphs
  

In [ ]:
# CLOSE
gs.close_connection()